# Semantic Federated — Experimentos no Colab

Notebook pensado para ser aberto direto pelo GitHub (**Arquivo > Abrir notebook > GitHub**, colando a URL do repositório `GianSE/semantic-federeted`).

Antes de rodar: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)**.

Fluxo das células:
1. Checar GPU
2. Clonar o repositório (o Colab abre só o `.ipynb`, não o resto do repo — por isso o clone é necessário)
3. Instalar dependências
4. Montar o Google Drive e redirecionar `data/` e `results/` para lá (persistência entre sessões, já que o disco do Colab é efêmero)
5. Rodar os experimentos
6. Conferir os resultados gerados

In [ ]:
!nvidia-smi

Se a célula acima der erro (`command not found` / `no devices found`), o ambiente de execução não está configurado como GPU — ajuste em **Ambiente de execução > Alterar tipo de ambiente de execução** antes de continuar.

In [ ]:
REPO_URL = "https://github.com/GianSE/semantic-federeted.git"
REPO_DIR = "/content/semantic-federeted"

import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositorio ja clonado, pulando.")

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Persistência via Google Drive

O disco do Colab é apagado quando a sessão termina. Para não perder os datasets baixados (evita re-download a cada sessão) nem os resultados acumulados (`results/`), apontamos essas duas pastas para o seu Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/semantic-federeted"
os.makedirs(f"{DRIVE_BASE}/data", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results", exist_ok=True)

import shutil

for local_name, drive_path in [("data", f"{DRIVE_BASE}/data"), ("results", f"{DRIVE_BASE}/results")]:
    local_path = os.path.join(REPO_DIR, local_name)
    if os.path.islink(local_path):
        continue
    if os.path.isdir(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f"{local_name}/ -> {drive_path}")

## Rodar os experimentos

O `main.py` já salva os resultados incrementalmente (um `save_results` por configuração), então se a sessão cair no meio, os resultados já salvos continuam no Drive e você pode rodar novamente depois — o `save_results` acumula no CSV/JSON existente em vez de sobrescrever.

Ajuste os argumentos conforme a necessidade (ex.: múltiplas seeds, mais rounds/clientes agora que tem GPU disponível).

In [ ]:
!python main.py \
  --datasets cifar10 \
  --latent-dims 16 32 64 128 \
  --noise-levels 0.0 0.01 0.05 0.1 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 42 \
  --num-workers 2

### Opcional: múltiplas seeds, não-IID e canais com desvanecimento

Agora que a GPU reduz bastante o tempo por execução, vale a pena rodar com múltiplas seeds para reportar média ± desvio padrão (a crítica mais repetida pelos revisores). O resultado agregado fica em `results/tables/results_summary.csv`.

In [ ]:
# Exemplo: 5 seeds, partição não-IID (Dirichlet) e canal com desvanecimento Rayleigh.
# Ajuste livremente -- nao precisa rodar essa celula se a anterior ja for suficiente.
!python main.py \
  --datasets cifar10 \
  --latent-dims 64 \
  --noise-levels 0.0 0.05 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 1 2 3 4 5 \
  --partition dirichlet \
  --dirichlet-alpha 0.3 \
  --channel-type rayleigh \
  --fading-scale 1.0 \
  --num-workers 2

## Conferir resultados

In [ ]:
import pandas as pd

df = pd.read_csv("results/data/experiment_results.csv")
df

Os gráficos (`results/plots/`) e tabelas LaTeX (`results/tables/`) ficam disponíveis no Google Drive em `MyDrive/semantic-federeted/results/`, prontos para serem baixados ou referenciados direto no paper.